# FRED Data Explorer — one-click portal

This is the whole app running inside Google Colab — no local install required. Anyone with the
link and a free Google account can open it and run it.

## To launch: Runtime menu -> Run all (or Ctrl+F9 / Cmd+F9)

That single action runs every cell below in order: installs dependencies, writes out the app,
asks for your FRED API key, starts the app, and opens a public link to it. The only manual steps
are pasting your API key when the box appears partway through, and opening the printed link at
the end.

Leave the last cell running - it's the tunnel keeping your link alive. Stop it when you're done
(the square stop button, or Runtime -> Interrupt execution), or just close the tab.


In [ ]:
!pip -q install streamlit fredapi keyring kaleido


### Your FRED API key
Get a free one at https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
import getpass
import os

os.environ["FRED_API_KEY"] = getpass.getpass("Paste your FRED API key: ")


### App files
These write out the same `app.py` / `fred_data.py` / theme you've been using locally.

In [ ]:
%%writefile fred_data.py
"""
Fetch and manipulate multiple FRED (Federal Reserve Economic Data) series.

Setup (one time):
    1. Get a free API key: https://fredapi.stlouisfed.org/docs/api/api_key.html
    2. Store it once: python setup_api_key.py
       (saved to Windows Credential Manager via `keyring` - no env var needed again)

Usage:
    python fred_data.py
"""

import os
import pandas as pd
import keyring
from fredapi import Fred

KEYRING_SERVICE = "fred_api"
KEYRING_USERNAME = "api_key"


def get_api_key() -> str | None:
    """Look for the key in, in order: env var, then OS credential store."""
    return os.environ.get("FRED_API_KEY") or keyring.get_password(KEYRING_SERVICE, KEYRING_USERNAME)


def get_recession_series() -> pd.Series:
    """NBER recession indicator (USREC): 1 during a recession month, 0 otherwise."""
    api_key = get_api_key()
    if not api_key:
        raise ValueError("No FRED API key found. Run `python setup_api_key.py` once to store your key.")
    return Fred(api_key=api_key).get_series("USREC")


def shade_recessions(ax, recession_series: pd.Series):
    """Shade NBER recession periods on a matplotlib Axes with a date x-axis."""
    xlim = ax.get_xlim()
    starts, ends = [], []
    in_rec = False
    for date, val in recession_series.items():
        if val == 1 and not in_rec:
            starts.append(date)
            in_rec = True
        elif val == 0 and in_rec:
            ends.append(date)
            in_rec = False
    if in_rec:
        ends.append(recession_series.index[-1])
    for s, e in zip(starts, ends):
        ax.axvspan(s, e, color="gray", alpha=0.15, zorder=0)
    ax.set_xlim(xlim)


class FredDatasets:
    """Fetch multiple FRED series and manipulate them together as one DataFrame."""

    def __init__(self, api_key: str | None = None):
        api_key = api_key or get_api_key()
        if not api_key:
            raise ValueError(
                "No FRED API key found. Run `python setup_api_key.py` once to store your key, "
                "or set the FRED_API_KEY environment variable, or pass api_key= explicitly."
            )
        self.fred = Fred(api_key=api_key)
        self.series: dict[str, pd.Series] = {}

    def add_series(self, series_id: str, label: str | None = None,
                    start: str | None = None, end: str | None = None) -> pd.Series:
        """Fetch one FRED series (e.g. 'GDP', 'UNRATE', 'CPIAUCSL') and store it under `label`."""
        label = label or series_id
        s = self.fred.get_series(series_id, observation_start=start, observation_end=end)
        s.name = label
        self.series[label] = s
        return s

    def add_many(self, series_map: dict[str, str],
                  start: str | None = None, end: str | None = None) -> pd.DataFrame:
        """series_map: {series_id: label}. Fetches each and returns the combined frame."""
        for series_id, label in series_map.items():
            self.add_series(series_id, label, start=start, end=end)
        return self.as_dataframe()

    def as_dataframe(self) -> pd.DataFrame:
        """Combine all fetched series into one DataFrame, aligned by date (outer join)."""
        if not self.series:
            return pd.DataFrame()
        return pd.concat(self.series.values(), axis=1)

    # --- manipulation helpers -------------------------------------------------

    def resample(self, freq: str = "ME", how: str = "mean") -> pd.DataFrame:
        """Resample combined data to a new frequency, e.g. 'ME' (month), 'QE' (quarter), 'YE' (year)."""
        df = self.as_dataframe()
        resampled = df.resample(freq)
        return getattr(resampled, how)()

    def pct_change(self, periods: int = 1) -> pd.DataFrame:
        """Percent change over `periods` observations, per column."""
        return self.as_dataframe().pct_change(periods=periods) * 100

    def yoy_change(self, freq: str = "ME") -> pd.DataFrame:
        """Year-over-year percent change, assuming monthly-ish data (12 periods back)."""
        df = self.resample(freq)
        return df.pct_change(periods=12) * 100

    def normalize(self, base_date: str | None = None) -> pd.DataFrame:
        """Rebase every series to 100 at `base_date` (or the first common non-NaN row)."""
        df = self.as_dataframe()
        if base_date:
            base = df.loc[base_date]
        else:
            base = df.dropna().iloc[0]
        return df.div(base) * 100

    def correlation(self) -> pd.DataFrame:
        """Correlation matrix across all series (pairwise, on overlapping dates)."""
        return self.as_dataframe().corr()

    def dropna_common(self) -> pd.DataFrame:
        """Rows where every series has data (inner-join equivalent)."""
        return self.as_dataframe().dropna()

    def to_csv(self, path: str):
        self.as_dataframe().to_csv(path)
        print(f"Saved {len(self.series)} series to {path}")


if __name__ == "__main__":
    from pathlib import Path

    fd = FredDatasets()

    # Example: unemployment rate, CPI, and real GDP
    fd.add_many(
        {
            "UNRATE": "Unemployment Rate",
            "CPIAUCSL": "CPI",
            "GDP": "Real GDP",
        },
        start="2000-01-01",
    )

    df = fd.as_dataframe()
    print("\nRaw data (tail):")
    print(df.tail())

    print("\nYear-over-year % change (tail):")
    print(fd.yoy_change().tail())

    print("\nCorrelation matrix:")
    print(fd.correlation())

    out_path = Path(__file__).resolve().parent / "fred_data.csv"
    fd.to_csv(str(out_path))


In [ ]:
%%writefile app.py
"""
Interactive FRED data explorer.

Run with:
    streamlit run app.py

Requires a FRED API key already stored via `python setup_api_key.py`
(or set as the FRED_API_KEY environment variable).
"""

import datetime
import io
import re

import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from fred_data import FredDatasets, get_api_key, get_recession_series, shade_recessions

st.set_page_config(page_title="FRED Data Explorer", layout="wide")

DEFAULT_PALETTE = px.colors.qualitative.Plotly  # cycled through when picking default line colors
IMAGE_FORMATS = {"PNG": "image/png", "JPEG": "image/jpeg", "SVG": "image/svg+xml", "PDF": "application/pdf"}
COLORSCALES = ["Viridis", "Plasma", "Turbo", "Cividis", "Blues", "Reds", "Greens", "Rainbow", "Bluered"]
MPL_FORMATS = {"PNG": "image/png", "PDF": "application/pdf", "SVG": "image/svg+xml"}

# A handful of well-known series to start from; users can add any FRED series ID.
DEFAULT_SERIES = {
    "UNRATE": "Unemployment Rate",
    "CPIAUCSL": "CPI (All Urban Consumers)",
    "GDP": "Real GDP",
    "FEDFUNDS": "Federal Funds Rate",
    "DGS10": "10-Year Treasury Yield",
    "PAYEMS": "Nonfarm Payrolls",
    "M2SL": "M2 Money Supply",
    "HOUST": "Housing Starts",
    "JTSJOR": "Job Openings Rate",
}

TRANSFORMS = ["Raw", "Year-over-year % change", "Period % change", "Normalized (=100 at start)"]

TIME_RANGE_OPTIONS = ["1Y", "5Y", "10Y", "20Y", "YTD", "Max"]

# Display name -> (plotly symbol, matplotlib marker), for manually-grouped scatter points
SHAPE_OPTIONS = {
    "Circle": ("circle", "o"),
    "Square": ("square", "s"),
    "Diamond": ("diamond", "D"),
    "Triangle": ("triangle-up", "^"),
    "Star": ("star", "*"),
    "Cross": ("x", "x"),
}

FREQ_MAP = {"None": None, "Monthly": "ME", "Quarterly": "QE", "Yearly": "YE"}

# X series, Y series, X transform, Y transform
RELATIONSHIP_PRESETS = {
    "Beveridge Curve (Unemployment vs Job Openings)": ("UNRATE", "JTSJOR", "Raw", "Raw"),
    "Phillips Curve (Unemployment vs Inflation)": ("UNRATE", "CPIAUCSL", "Raw", "Year-over-year % change"),
    "Custom": None,
}

# Operations available when combining two series into a derived series (e.g. V/E ratio = A / B)
DERIVED_OPS = {
    "A / B (ratio)": (lambda a, b: a / b, "/"),
    "A - B (difference)": (lambda a, b: a - b, "-"),
    "A + B (sum)": (lambda a, b: a + b, "+"),
    "A * B (product)": (lambda a, b: a * b, "*"),
    "A as % of B": (lambda a, b: a / b * 100, "% of"),
}

DERIVED_PRESETS = {
    "None": None,
    "V/E Ratio (Job Openings Rate / Unemployment Rate)": ("JTSJOR", "A / B (ratio)", "UNRATE", "V/E Ratio"),
}

st.title("FRED Data Explorer")

if not get_api_key():
    st.error(
        "No FRED API key found. Run `python setup_api_key.py` in this folder once, "
        "then reload this page."
    )
    st.stop()


@st.cache_data(show_spinner="Fetching from FRED...")
def fetch(series_ids: tuple[str, ...], start: str, end: str) -> pd.DataFrame:
    fd = FredDatasets()
    for sid in series_ids:
        label = DEFAULT_SERIES.get(sid, sid)
        fd.add_series(sid, label, start=start, end=end)
    return fd.as_dataframe()


def slugify(name: str) -> str:
    """Turn a display name like 'V/E Ratio' into a stable, typeable ID like 'V_E_RATIO'."""
    slug = re.sub(r"[^A-Za-z0-9]+", "_", name.strip()).strip("_").upper()
    return slug or "SERIES"


def get_derived_series() -> dict:
    return st.session_state.setdefault("derived_series", {})


def series_display_label(sid: str) -> str:
    """Human-readable name for a series ID, whether it's a base FRED ID or a derived series."""
    derived = get_derived_series()
    if sid in derived:
        return derived[sid]["display"]
    return DEFAULT_SERIES.get(sid, sid)


def fetch_with_derived(selected_ids: tuple[str, ...], start: str, end: str) -> pd.DataFrame:
    """Like `fetch`, but transparently resolves any derived-series IDs into their formula
    over the underlying base FRED series."""
    derived = get_derived_series()
    base_needed = []
    for sid in selected_ids:
        if sid in derived:
            base_needed += [derived[sid]["a_id"], derived[sid]["b_id"]]
        else:
            base_needed.append(sid)
    base_needed = tuple(dict.fromkeys(base_needed))  # dedupe, keep order

    base_df = fetch(base_needed, start, end)

    out = pd.DataFrame(index=base_df.index)
    for sid in selected_ids:
        label = series_display_label(sid)
        if sid in derived:
            d = derived[sid]
            a_col, b_col = DEFAULT_SERIES.get(d["a_id"], d["a_id"]), DEFAULT_SERIES.get(d["b_id"], d["b_id"])
            op_fn, _ = DERIVED_OPS[d["op"]]
            out[label] = op_fn(base_df[a_col], base_df[b_col])
        else:
            out[label] = base_df[label]
    return out


def contrast_text_color(hex_color: str) -> str:
    """Pick black or white text so it stays legible against `hex_color`."""
    hex_color = hex_color.lstrip("#")
    r, g, b = (int(hex_color[i : i + 2], 16) for i in (0, 2, 4))
    luminance = 0.299 * r + 0.587 * g + 0.114 * b
    return "#000000" if luminance > 140 else "#FFFFFF"


def compute_preset_range(preset: str) -> tuple[datetime.date, datetime.date]:
    today = datetime.date.today()
    if preset == "YTD":
        return datetime.date(today.year, 1, 1), today
    if preset == "Max":
        return datetime.date(1900, 1, 1), today  # earlier than any FRED series
    years = int(preset.rstrip("Y"))
    return today - datetime.timedelta(days=365 * years), today


def time_window_control(key_prefix: str, start_key: str, end_key: str):
    """A row of quick-range buttons (1Y/5Y/10Y/.../Max) that set the Start/End
    date_input widgets below when clicked. Applies only on the click itself (not
    every rerun), so a manual edit to Start/End afterward isn't clobbered."""
    quick_range = st.segmented_control(
        "Quick range", options=TIME_RANGE_OPTIONS, key=f"{key_prefix}_quick_range"
    )
    prev_key = f"{key_prefix}_quick_range_prev"
    if quick_range and quick_range != st.session_state.get(prev_key):
        preset_start, preset_end = compute_preset_range(quick_range)
        st.session_state[start_key] = preset_start
        st.session_state[end_key] = preset_end
    st.session_state[prev_key] = quick_range


def apply_transform(series: pd.Series, transform: str, periods: int) -> tuple[pd.Series, str]:
    if transform == "Year-over-year % change":
        return series.pct_change(periods=periods) * 100, "YoY % change"
    if transform == "Period % change":
        return series.pct_change() * 100, "% change"
    if transform == "Normalized (=100 at start)":
        clean = series.dropna()
        base = clean.iloc[0] if not clean.empty else series.iloc[0]
        return series.div(base) * 100, "Index (start = 100)"
    return series, "Value"


def export_chart_section(fig: go.Figure, key_prefix: str, file_stem: str):
    """Render format/size controls and a download button that rasterizes `fig` via kaleido."""
    st.markdown("**Export chart as image**")
    c1, c2, c3, c4 = st.columns(4)
    with c1:
        fmt = st.selectbox("Format", list(IMAGE_FORMATS.keys()), key=f"{key_prefix}_fmt")
    with c2:
        width = st.number_input("Width (px)", min_value=200, max_value=4000, value=1200, step=100, key=f"{key_prefix}_w")
    with c3:
        height = st.number_input("Height (px)", min_value=200, max_value=4000, value=700, step=100, key=f"{key_prefix}_h")
    with c4:
        scale = st.number_input("Scale", min_value=1.0, max_value=5.0, value=2.0, step=0.5, key=f"{key_prefix}_scale")

    if st.button("Generate image", key=f"{key_prefix}_gen"):
        try:
            img_bytes = fig.to_image(format=fmt.lower(), width=int(width), height=int(height), scale=scale)
            st.session_state[f"{key_prefix}_img"] = img_bytes
        except Exception as e:
            st.error(
                f"Couldn't render the image ({e}). Kaleido needs a local Chromium; "
                "try running `kaleido_get_chrome` once in this environment."
            )

    img_bytes = st.session_state.get(f"{key_prefix}_img")
    if img_bytes:
        st.download_button(
            f"Download {fmt}",
            img_bytes,
            file_name=f"{file_stem}.{fmt.lower()}",
            mime=IMAGE_FORMATS[fmt],
            key=f"{key_prefix}_dl",
        )


@st.cache_data(show_spinner="Fetching NBER recession dates...")
def fetch_recessions() -> pd.Series:
    return get_recession_series()


def pick_year_interval(start, end) -> int:
    """A tick every N years, scaled so long histories don't get an unreadable axis."""
    years = (end - start).days / 365.25
    if years > 60:
        return 20
    if years > 30:
        return 10
    if years > 15:
        return 5
    if years > 6:
        return 2
    return 1


def matplotlib_fig_to_bytes(fig, fmt: str) -> bytes:
    buf = io.BytesIO()
    fig.savefig(buf, format=fmt.lower(), dpi=200, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()


def build_panels_figure(plot_df: pd.DataFrame, label_colors: dict, y_label: str, recession_series):
    """One subplot per series, each with a bold title colored to match its line — see the
    Unemployment/Core PCE sample chart."""
    n = len(plot_df.columns)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 3))
    axes = [axes] if n == 1 else list(axes)
    interval = pick_year_interval(plot_df.index.min(), plot_df.index.max())
    for ax, col in zip(axes, plot_df.columns):
        color = label_colors.get(col, "#00205B")
        series = plot_df[col].dropna()
        ax.plot(series.index, series.values, color=color, linewidth=1.5)
        ax.set_title(f"{col} ({y_label})", fontsize=9, fontweight="bold", color=color)
        ax.tick_params(labelsize=6)
        ax.xaxis.set_major_locator(mdates.YearLocator(interval))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.grid(alpha=0.3)
        if recession_series is not None:
            shade_recessions(ax, recession_series)
    fig.tight_layout()
    return fig


def build_overlay_figure(
    plot_df: pd.DataFrame, label_colors: dict, y_label: str, recession_series, secondary_cols: list | None = None
):
    """All series overlaid on one panel with a legend — see the Inflation Indicators sample
    chart. Series in `secondary_cols` are plotted against a right-hand y-axis (twinx), so
    curves with very different scales/units can still be overlaid meaningfully."""
    secondary_cols = [c for c in (secondary_cols or []) if c in plot_df.columns]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax2 = ax.twinx() if secondary_cols else None
    interval = pick_year_interval(plot_df.index.min(), plot_df.index.max())

    lines = []
    for col in plot_df.columns:
        target = ax2 if (ax2 is not None and col in secondary_cols) else ax
        series = plot_df[col].dropna()
        (line,) = target.plot(series.index, series.values, color=label_colors.get(col), linewidth=1.5, label=col)
        lines.append(line)

    title = plot_df.columns[0] if len(plot_df.columns) == 1 else f"FRED Series ({y_label})"
    ax.set_title(title, fontsize=11, fontweight="bold")
    primary_cols = [c for c in plot_df.columns if c not in secondary_cols]
    ax.set_ylabel(", ".join(primary_cols) if len(primary_cols) <= 2 else y_label, fontsize=8)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mdates.YearLocator(interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(alpha=0.3)
    if ax2 is not None:
        ax2.set_ylabel(", ".join(secondary_cols) if len(secondary_cols) <= 2 else "Secondary axis", fontsize=8)
        ax2.tick_params(labelsize=7)
    if len(plot_df.columns) > 1:
        ax.legend(lines, [l.get_label() for l in lines], fontsize=7, loc="upper left")
    if recession_series is not None:
        shade_recessions(ax, recession_series)
    fig.tight_layout()
    return fig


def build_relationship_publication_fig(
    merged: pd.DataFrame, x_label: str, y_label_name: str, point_color: str,
    highlight_color: str, highlight_latest: bool, show_label: bool,
    point_groups: list | None = None, groups: dict | None = None,
):
    """Muted scatter with the latest point called out — see the Beveridge Curve sample chart.
    If `point_groups` (a group name or None per row of `merged`) and `groups` (name -> {color,
    shape}) are given, manually-assigned points are drawn with their group's color/marker and
    get a legend; everything else stays the plain muted scatter."""
    fig, ax = plt.subplots(figsize=(6, 4.5))
    groups = groups or {}
    if point_groups:
        ungrouped_mask = [g not in groups for g in point_groups]
        ax.scatter(
            merged["x"][ungrouped_mask], merged["y"][ungrouped_mask],
            color=point_color, alpha=0.6, s=25, edgecolor="none", zorder=2,
        )
        for gname, gcfg in groups.items():
            mask = [g == gname for g in point_groups]
            if any(mask):
                _, marker = SHAPE_OPTIONS.get(gcfg["shape"], ("circle", "o"))
                ax.scatter(
                    merged["x"][mask], merged["y"][mask],
                    color=gcfg["color"], marker=marker, s=40, edgecolor="none", zorder=3, label=gname,
                )
        ax.legend(fontsize=7, loc="best")
    else:
        ax.scatter(merged["x"], merged["y"], color=point_color, alpha=0.6, s=25, edgecolor="none", zorder=2)
    if highlight_latest:
        lx, ly = merged["x"].iloc[-1], merged["y"].iloc[-1]
        ax.scatter([lx], [ly], color=highlight_color, s=45, zorder=4)
        if show_label:
            ax.annotate(
                merged.index[-1].strftime("%Y-%m"),
                (lx, ly),
                textcoords="offset points",
                xytext=(6, 6),
                fontsize=8,
                color=highlight_color,
            )
    ax.set_title(f"{x_label} vs. {y_label_name}", fontsize=11, fontweight="bold")
    ax.set_xlabel(x_label, fontsize=9)
    ax.set_ylabel(y_label_name, fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def publication_export_section(key_prefix: str, file_stem: str, build_fn, needs_recession: bool = False):
    """Shared controls for the matplotlib 'publication style' export, used by both tabs."""
    recession_series = None
    if needs_recession:
        shade = st.checkbox("Shade NBER recessions", value=True, key=f"{key_prefix}_shade")
        if shade:
            try:
                recession_series = fetch_recessions()
            except Exception as e:
                st.warning(f"Couldn't load recession dates: {e}")

    fmt = st.selectbox("Format", list(MPL_FORMATS.keys()), key=f"{key_prefix}_fmt")

    if st.button("Generate image", key=f"{key_prefix}_gen"):
        try:
            fig = build_fn(recession_series) if needs_recession else build_fn()
            st.session_state[f"{key_prefix}_img"] = matplotlib_fig_to_bytes(fig, fmt)
        except Exception as e:
            st.error(f"Couldn't render the image: {e}")

    img_bytes = st.session_state.get(f"{key_prefix}_img")
    if img_bytes:
        if fmt == "PNG":
            st.image(img_bytes)
        st.download_button(
            f"Download {fmt}",
            img_bytes,
            file_name=f"{file_stem}.{fmt.lower()}",
            mime=MPL_FORMATS[fmt],
            key=f"{key_prefix}_dl",
        )


with st.sidebar:
    st.header("Derived series")
    with st.expander("Create a derived series"):
        st.caption(
            "Combine two FRED series with a formula, e.g. **V/E Ratio** = Job Openings Rate / Unemployment Rate — "
            "a common labor-market tightness measure."
        )
        derived_preset = st.selectbox("Quick preset", list(DERIVED_PRESETS.keys()), key="derived_preset")
        preset_vals = DERIVED_PRESETS[derived_preset]
        d_a_default, d_op_default, d_b_default, d_name_default = preset_vals or ("UNRATE", "A / B (ratio)", "CPIAUCSL", "")

        d_a_id = st.text_input("Series A ID", value=d_a_default, key=f"d_a_{derived_preset}").strip().upper()
        d_op = st.selectbox(
            "Operation", list(DERIVED_OPS.keys()), index=list(DERIVED_OPS.keys()).index(d_op_default), key=f"d_op_{derived_preset}"
        )
        d_b_id = st.text_input("Series B ID", value=d_b_default, key=f"d_b_{derived_preset}").strip().upper()
        d_name = st.text_input("Name for the new series", value=d_name_default, key=f"d_name_{derived_preset}")

        if st.button("Add derived series", key="add_derived"):
            if d_name.strip() and d_a_id and d_b_id:
                slug = slugify(d_name)
                get_derived_series()[slug] = {"display": d_name.strip(), "a_id": d_a_id, "op": d_op, "b_id": d_b_id}
                st.success(f"Added **{d_name.strip()}** — pick it from 'Pick series' below, or type `{slug}` as an X/Y series ID.")
            else:
                st.warning("Give it a name and both series IDs.")

    derived_defs = get_derived_series()
    if derived_defs:
        st.caption("Your derived series:")
        for slug, d in list(derived_defs.items()):
            c1, c2 = st.columns([5, 1])
            _, symbol = DERIVED_OPS[d["op"]]
            c1.write(f"**{d['display']}** (`{slug}`) = {d['a_id']} {symbol} {d['b_id']}")
            if c2.button("✕", key=f"del_{slug}", help="Remove"):
                del derived_defs[slug]
                st.rerun()

tab_series, tab_relationship = st.tabs(["Time Series", "Relationship (X vs Y)"])

# --- Time Series tab -----------------------------------------------------

with tab_series:
    with st.sidebar:
        st.header("Series")
        series_options = list(DEFAULT_SERIES.keys()) + list(derived_defs.keys())
        chosen_labels = st.multiselect(
            "Pick series",
            options=series_options,
            default=["UNRATE", "CPIAUCSL"],
            format_func=lambda sid: f"🧮 {derived_defs[sid]['display']}" if sid in derived_defs else f"{sid} — {DEFAULT_SERIES[sid]}",
        )
        custom = st.text_input("Add a custom FRED series ID (comma-separated)", "")
        custom_ids = [c.strip().upper() for c in custom.split(",") if c.strip()]
        series_ids = tuple(dict.fromkeys(chosen_labels + custom_ids))  # dedupe, keep order

        st.header("Date range")
        time_window_control("ts", "ts_start", "ts_end")
        start_date = st.date_input(
            "Start", value=datetime.date(2000, 1, 1),
            min_value=datetime.date(1900, 1, 1), max_value=datetime.date.today(), key="ts_start",
        )
        end_date = st.date_input(
            "End", value=datetime.date.today(),
            min_value=datetime.date(1900, 1, 1), max_value=datetime.date.today(), key="ts_end",
        )

        st.header("Transform")
        view = st.radio("View data as", TRANSFORMS, key="ts_view")
        freq_label = st.selectbox("Resample frequency", list(FREQ_MAP.keys()), key="ts_freq")
        freq = FREQ_MAP[freq_label]

        st.header("Colors")
        ts_bg_color = st.color_picker("Background", "#FFFFFF", key="ts_bg")
        series_colors = {}
        for i, sid in enumerate(series_ids):
            default_color = DEFAULT_PALETTE[i % len(DEFAULT_PALETTE)]
            series_colors[sid] = st.color_picker(
                series_display_label(sid), default_color, key=f"ts_color_{sid}"
            )

    if not series_ids:
        st.info("Pick at least one series from the sidebar.")
        st.stop()

    df = fetch_with_derived(series_ids, start_date.isoformat(), end_date.isoformat())

    if freq:
        df = df.resample(freq).mean()

    periods = 12 if freq in (None, "ME") else (4 if freq == "QE" else 1)
    plot_df = pd.DataFrame({col: apply_transform(df[col], view, periods)[0] for col in df.columns})
    y_label = apply_transform(df[df.columns[0]], view, periods)[1]
    plot_df = plot_df.dropna(how="all")

    label_colors = {series_display_label(sid): series_colors[sid] for sid in series_ids}
    ts_text_color = contrast_text_color(ts_bg_color)

    st.subheader("Chart")
    fig = go.Figure()
    for col in plot_df.columns:
        fig.add_trace(
            go.Scatter(x=plot_df.index, y=plot_df[col], mode="lines", name=col, line=dict(color=label_colors.get(col)))
        )
    fig.update_layout(
        height=500,
        yaxis_title=y_label,
        xaxis_title="Date",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=20, t=40, b=40),
        plot_bgcolor=ts_bg_color,
        paper_bgcolor=ts_bg_color,
        font=dict(color=ts_text_color),
    )
    st.plotly_chart(fig, use_container_width=True)

    with st.expander("Export chart as image (interactive style)"):
        export_chart_section(fig, "ts_export", "fred_timeseries")

    with st.expander("Publication-style export (matplotlib, NBER shading)"):
        layout = st.radio(
            "Layout",
            ["Side-by-side panels", "Single overlay"],
            index=0 if len(plot_df.columns) <= 2 else 1,
            horizontal=True,
            key="ts_pub_layout",
        )
        if layout == "Side-by-side panels":
            build_fn = lambda rec: build_panels_figure(plot_df, label_colors, y_label, rec)
        else:
            secondary_cols = st.multiselect(
                "Series on secondary (right) axis — use this when curves have very different scales/units",
                options=list(plot_df.columns),
                key="ts_pub_secondary",
            )
            build_fn = lambda rec: build_overlay_figure(plot_df, label_colors, y_label, rec, secondary_cols)
        publication_export_section("ts_pub", "fred_timeseries_publication", build_fn, needs_recession=True)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Data")
        st.dataframe(plot_df, use_container_width=True, height=350)
        st.download_button(
            "Download CSV",
            plot_df.to_csv().encode("utf-8"),
            file_name="fred_export.csv",
            mime="text/csv",
        )

    with col2:
        if len(series_ids) > 1:
            st.subheader("Correlation")
            corr = df.corr()
            heat = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
            heat.update_layout(height=350, margin=dict(l=20, r=20, t=20, b=20))
            st.plotly_chart(heat, use_container_width=True)
        else:
            st.subheader("Summary stats")
            st.dataframe(df.describe(), use_container_width=True)

# --- Relationship (X vs Y) tab --------------------------------------------

with tab_relationship:
    st.write(
        "Plot one series against another over time — e.g. the **Beveridge curve** "
        "(unemployment vs. job openings) or the **Phillips curve** (unemployment vs. inflation). "
        "Points are connected in chronological order and colored by date, or you can manually "
        "select and group points with your own colors and shapes (see below the chart)."
    )

    preset_choice = st.selectbox("Preset", list(RELATIONSHIP_PRESETS.keys()), key="rel_preset")
    preset = RELATIONSHIP_PRESETS[preset_choice]
    default_x, default_y, default_xt, default_yt = preset if preset else ("UNRATE", "CPIAUCSL", "Raw", "Raw")

    if derived_defs:
        st.caption("You can also type a derived series ID here, e.g. `" + next(iter(derived_defs)) + "`.")

    # Keying widgets by the preset forces them to pick up new defaults when the
    # preset changes, while still letting the user freely edit within a preset.
    colA, colB = st.columns(2)
    with colA:
        st.markdown("**X axis**")
        x_id = st.text_input("X series ID", value=default_x, key=f"x_id_{preset_choice}").strip().upper()
        x_transform = st.selectbox(
            "X transform", TRANSFORMS, index=TRANSFORMS.index(default_xt), key=f"x_t_{preset_choice}"
        )
    with colB:
        st.markdown("**Y axis**")
        y_id = st.text_input("Y series ID", value=default_y, key=f"y_id_{preset_choice}").strip().upper()
        y_transform = st.selectbox(
            "Y transform", TRANSFORMS, index=TRANSFORMS.index(default_yt), key=f"y_t_{preset_choice}"
        )

    time_window_control("rel", "rel_start", "rel_end")
    colC, colD, colE = st.columns(3)
    with colC:
        rel_start = st.date_input(
            "Start", value=datetime.date(2000, 1, 1),
            min_value=datetime.date(1900, 1, 1), max_value=datetime.date.today(), key="rel_start",
        )
    with colD:
        rel_end = st.date_input(
            "End", value=datetime.date.today(),
            min_value=datetime.date(1900, 1, 1), max_value=datetime.date.today(), key="rel_end",
        )
    with colE:
        rel_freq_label = st.selectbox("Resample frequency", list(FREQ_MAP.keys()), index=1, key="rel_freq")
        rel_freq = FREQ_MAP[rel_freq_label]

    st.markdown("**Colors**")
    colF, colG, colH, colI = st.columns(4)
    with colF:
        rel_line_color = st.color_picker("Path line", "#A0A0A0", key="rel_line_color")
    with colG:
        rel_colorscale = st.selectbox("Point gradient (by year)", COLORSCALES, key="rel_colorscale")
    with colH:
        rel_latest_color = st.color_picker("Latest point", "#FF0000", key="rel_latest_color")
    with colI:
        rel_bg_color = st.color_picker("Background", "#FFFFFF", key="rel_bg_color")
    rel_point_color = st.color_picker(
        "Publication-style point color (used in the matplotlib export below)", "#4C72B0", key="rel_point_color"
    )

    groups = st.session_state.setdefault("rel_groups", {})
    assignments = st.session_state.setdefault("rel_point_assignments", {})

    with st.expander("Manually segregate points (color/shape by group)"):
        st.caption(
            "Box- or lasso-select points on the chart below, then assign them to a named group. "
            "Grouped points get their own color and marker shape, in place of the year-gradient coloring."
        )
        gc1, gc2, gc3, gc4 = st.columns([2, 1, 1, 1])
        with gc1:
            new_group_name = st.text_input("Group name", key="rel_new_group_name")
        with gc2:
            new_group_color = st.color_picker(
                "Color", DEFAULT_PALETTE[len(groups) % len(DEFAULT_PALETTE)], key="rel_new_group_color"
            )
        with gc3:
            new_group_shape = st.selectbox("Shape", list(SHAPE_OPTIONS.keys()), key="rel_new_group_shape")
        with gc4:
            st.write("")
            st.write("")
            if st.button("Add group", key="rel_add_group"):
                if new_group_name.strip():
                    groups[new_group_name.strip()] = {"color": new_group_color, "shape": new_group_shape}
                else:
                    st.warning("Give the group a name.")

        if groups:
            st.caption("Your groups:")
            for gname, gcfg in list(groups.items()):
                r1, r2, r3 = st.columns([2, 2, 1])
                r1.color_picker(gname, gcfg["color"], key=f"rel_group_swatch_{gname}", disabled=True)
                r2.write(f"Shape: {gcfg['shape']}")
                if r3.button("Remove", key=f"rel_del_group_{gname}"):
                    del groups[gname]
                    for k in [k for k, v in assignments.items() if v == gname]:
                        del assignments[k]
                    st.rerun()

    if x_id and y_id:
        rel_df = fetch_with_derived((x_id, y_id), rel_start.isoformat(), rel_end.isoformat())

        if rel_df.shape[1] < 2:
            st.error("Couldn't fetch both series — check that both FRED (or derived) series IDs are valid.")
        else:
            if rel_freq:
                rel_df = rel_df.resample(rel_freq).mean()
            rel_periods = 12 if rel_freq in (None, "ME") else (4 if rel_freq == "QE" else 1)

            x_label = series_display_label(x_id)
            y_label_name = series_display_label(y_id)
            x_col, x_axis_label = apply_transform(rel_df[x_label], x_transform, rel_periods)
            y_col, y_axis_label = apply_transform(rel_df[y_label_name], y_transform, rel_periods)

            merged = pd.concat([x_col.rename("x"), y_col.rename("y")], axis=1).dropna()

            if merged.empty:
                st.warning("No overlapping data for these two series/date range.")
            else:
                st.subheader(f"{x_label} vs. {y_label_name}")
                year_frac = merged.index.year + (merged.index.month - 1) / 12
                rel_text_color = contrast_text_color(rel_bg_color)
                point_groups = [assignments.get(ts.isoformat()) for ts in merged.index]

                fig2 = go.Figure()
                # Background connecting line, drawn separately from the markers so box/lasso
                # selection (which only targets markers) isn't confused by line segments.
                fig2.add_trace(
                    go.Scatter(
                        x=merged["x"], y=merged["y"], mode="lines",
                        line=dict(width=1, color=rel_line_color), hoverinfo="skip", showlegend=False,
                    )
                )

                if groups:
                    marker_dict = dict(
                        size=9,
                        color=[groups[g]["color"] if g in groups else "#B0B0B0" for g in point_groups],
                        symbol=[SHAPE_OPTIONS[groups[g]["shape"]][0] if g in groups else "circle" for g in point_groups],
                        line=dict(width=0.5, color="white"),
                    )
                else:
                    marker_dict = dict(
                        size=7, color=year_frac, colorscale=rel_colorscale,
                        showscale=True, colorbar=dict(title="Year"),
                    )

                fig2.add_trace(
                    go.Scatter(
                        x=merged["x"], y=merged["y"], mode="markers",
                        marker=marker_dict,
                        text=[d.strftime("%Y-%m") for d in merged.index],
                        hovertemplate="%{text}<br>X=%{x:.2f}<br>Y=%{y:.2f}<extra></extra>",
                        showlegend=False,
                    )
                )
                # Mark the most recent point so the current position on the curve is obvious.
                fig2.add_trace(
                    go.Scatter(
                        x=[merged["x"].iloc[-1]],
                        y=[merged["y"].iloc[-1]],
                        mode="markers",
                        marker=dict(size=13, color=rel_latest_color, symbol="star"),
                        name="Latest",
                        hovertemplate=f"Latest: {merged.index[-1].strftime('%Y-%m')}<extra></extra>",
                    )
                )
                fig2.update_layout(
                    height=550,
                    xaxis_title=f"{x_label} ({x_axis_label})",
                    yaxis_title=f"{y_label_name} ({y_axis_label})",
                    showlegend=False,
                    margin=dict(l=40, r=20, t=40, b=40),
                    plot_bgcolor=rel_bg_color,
                    paper_bgcolor=rel_bg_color,
                    font=dict(color=rel_text_color),
                )
                event = st.plotly_chart(
                    fig2, use_container_width=True, key="rel_scatter_chart",
                    on_select="rerun", selection_mode=["points", "box", "lasso"],
                )

                selected_points = event.selection.points if event and event.selection else []
                selected_indices = [p["point_index"] for p in selected_points if p.get("curve_number") == 1]

                if groups:
                    sc1, sc2, sc3 = st.columns([2, 2, 1])
                    with sc1:
                        st.caption(f"{len(selected_indices)} point(s) selected (box/lasso/click on the chart).")
                    with sc2:
                        assign_choice = st.selectbox(
                            "Assign selection to", list(groups.keys()),
                            key="rel_assign_choice", label_visibility="collapsed",
                        )
                    with sc3:
                        if st.button("Assign", key="rel_assign_btn", disabled=not selected_indices):
                            for i in selected_indices:
                                assignments[merged.index[i].isoformat()] = assign_choice
                            st.rerun()
                    if assignments and st.button("Clear all group assignments", key="rel_clear_assignments"):
                        assignments.clear()
                        st.rerun()
                elif selected_indices:
                    st.caption(f"{len(selected_indices)} point(s) selected — define a group above to assign them.")

                st.markdown("**Or group by time**")
                data_min, data_max = merged.index.min().date(), merged.index.max().date()
                range_start, range_end = st.slider(
                    "Date range",
                    min_value=data_min, max_value=data_max, value=(data_min, data_max),
                    key="rel_time_group_range",
                )
                tc3, tc4 = st.columns([2, 1])
                with tc3:
                    range_group = st.selectbox(
                        "Assign range to group", list(groups.keys()),
                        key="rel_time_group_choice", disabled=not groups,
                    )
                with tc4:
                    st.write("")
                    if st.button("Assign range", key="rel_time_group_assign", disabled=not groups):
                        for ts in merged.index:
                            if range_start <= ts.date() <= range_end:
                                assignments[ts.isoformat()] = range_group
                        st.rerun()

                ac1, ac2 = st.columns([2, 1])
                with ac1:
                    auto_period = st.selectbox(
                        "...or auto-create one group per", ["Year", "Decade"], key="rel_auto_period"
                    )
                with ac2:
                    st.write("")
                    if st.button("Auto-group every point", key="rel_auto_group_btn"):
                        for ts in merged.index:
                            label = str(ts.year) if auto_period == "Year" else f"{(ts.year // 10) * 10}s"
                            if label not in groups:
                                groups[label] = {
                                    "color": DEFAULT_PALETTE[len(groups) % len(DEFAULT_PALETTE)],
                                    "shape": "Circle",
                                }
                            assignments[ts.isoformat()] = label
                        st.rerun()

                with st.expander("Export chart as image (interactive style)"):
                    export_chart_section(fig2, "rel_export", "fred_relationship")

                with st.expander("Publication-style export (matplotlib)"):
                    highlight_latest = st.checkbox("Highlight latest point", value=True, key="rel_pub_highlight")
                    show_label = st.checkbox("Show date label on latest point", value=True, key="rel_pub_label")
                    build_fn = lambda: build_relationship_publication_fig(
                        merged, x_label, y_label_name, rel_point_color, rel_latest_color,
                        highlight_latest, show_label, point_groups=point_groups, groups=groups,
                    )
                    publication_export_section(
                        "rel_pub", "fred_relationship_publication", build_fn, needs_recession=False
                    )

                st.download_button(
                    "Download CSV",
                    merged.rename(columns={"x": x_label, "y": y_label_name}).to_csv().encode("utf-8"),
                    file_name="fred_relationship.csv",
                    mime="text/csv",
                )


In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)


In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"


### Warm up (speeds up the first page load)

The first time Streamlit's page loads, matplotlib silently builds a font cache and Python compiles the heavy imports (pandas, plotly, matplotlib) — that alone can be 30-90s with nothing visible in the browser. Doing it here instead means it happens while you watch this cell run, not while staring at a blank tunnel tab.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go

fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1])
fig.savefig("/tmp/_warmup.png")
plt.close(fig)

print("Warmed up. The Streamlit subprocess launched below reuses this cache.")


### Launch

Starts Streamlit in the background, then opens a public tunnel to it. The first time you open the
printed URL, localtunnel shows an interstitial page asking for a **Tunnel Password** — paste in
the IP address printed by the cell below (that's normal, it's how localtunnel confirms you're not
a random visitor).


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true &>/content/logs.txt &

import time
time.sleep(5)  # give it a moment to bind the port before the tunnel cells below


In [ ]:
print("Tunnel password (paste this on the interstitial page):")
!wget -q -O - ipv4.icanhazip.com


In [ ]:
!npm install -q -g localtunnel
!npx localtunnel --port 8501


### Troubleshooting

- **Nothing prints from the tunnel cell / connection refused**: Streamlit may still be starting.
  Wait ~10s and re-run the `npx localtunnel` cell.
- **Check Streamlit's own logs**: `!cat /content/logs.txt`
- **Stop everything**: Runtime → Interrupt execution, or just close/restart the runtime.
